# 07 - Ataques Adversariales contra Modelos de IA

Este cuaderno implementa técnicas de ataque adversarial y estrategias de defensa para modelos de detección de malware.

**Contenido:**
- Tipos de ataques adversariales
- Generación de ejemplos adversariales con ruido gaussiano
- Evaluación del impacto del ataque
- Defensa mediante entrenamiento adversarial

## 8.1 Amenazas a los modelos de IA

Los propios modelos de IA pueden ser objetivo de ataques. Un atacante puede manipular las entradas del modelo para engañarlo y evadir la detección.

| Tipo | Descripción |
|---|---|
| **Evasión** | Perturbaciones mínimas en la entrada para que el modelo clasifique malware como benigno |
| **Envenenamiento** | Inyección de datos maliciosos en el conjunto de entrenamiento |
| **Inversión** | Reconstrucción de datos de entrenamiento a partir del modelo |
| **Extracción** | Réplica del modelo mediante consultas masivas |

## 8.2 Preparación del dataset y modelo base

In [ ]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# ---------------------------------------------------------------
# Generar dataset sintético si no existe
# ---------------------------------------------------------------
if not os.path.exists('file_features.csv'):
    print('Generando dataset sintético de características PE...')
    rng = np.random.default_rng(42)
    n_benign, n_malicious = 800, 200
    benign = pd.DataFrame({
        'entry_point'        : rng.integers(4096, 8192, n_benign),
        'image_base'         : rng.integers(0x400000, 0x500000, n_benign),
        'size_of_image'      : rng.integers(50000, 200000, n_benign),
        'size_code_section'  : rng.integers(10000, 80000, n_benign),
        'dll_flag'           : rng.integers(0, 256, n_benign),
        'num_sections'       : rng.integers(3, 7, n_benign),
        'entropia_max'       : rng.uniform(4.0, 6.5, n_benign),
        'entropia_media'     : rng.uniform(3.0, 5.5, n_benign),
        'num_importaciones'  : rng.integers(50, 200, n_benign),
        'num_dlls_importadas': rng.integers(3, 10, n_benign),
        'num_exportaciones'  : rng.integers(0, 20, n_benign),
        'file_size'          : rng.integers(50000, 500000, n_benign),
        'label'              : 0
    })
    malicious = pd.DataFrame({
        'entry_point'        : rng.integers(4096, 8192, n_malicious),
        'image_base'         : rng.integers(0x400000, 0x500000, n_malicious),
        'size_of_image'      : rng.integers(50000, 200000, n_malicious),
        'size_code_section'  : rng.integers(10000, 80000, n_malicious),
        'dll_flag'           : rng.integers(0, 256, n_malicious),
        'num_sections'       : rng.integers(5, 12, n_malicious),
        'entropia_max'       : rng.uniform(6.5, 8.0, n_malicious),
        'entropia_media'     : rng.uniform(5.5, 7.5, n_malicious),
        'num_importaciones'  : rng.integers(200, 500, n_malicious),
        'num_dlls_importadas': rng.integers(8, 20, n_malicious),
        'num_exportaciones'  : rng.integers(0, 5, n_malicious),
        'file_size'          : rng.integers(100000, 1000000, n_malicious),
        'label'              : 1
    })
    df_features = pd.concat([benign, malicious], ignore_index=True)
    df_features.to_csv('file_features.csv', index=False)
    print(f'Dataset sintético guardado: {len(df_features)} muestras.')

# 1. Datos y modelo base
df = pd.read_csv('file_features.csv').dropna()
X, y = df.drop('label', axis=1).values, df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Normalizar a [0, 1] para que el ruido tenga sentido
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

modelo = RandomForestClassifier(n_estimators=200, random_state=42)
modelo.fit(X_train, y_train)

acc_normal = accuracy_score(y_test, modelo.predict(X_test))
print(f'Modelo base entrenado.')
print(f'Precisión normal: {acc_normal:.4f}')

## 8.3 Generación de ejemplos adversariales

In [ ]:
def generar_ejemplos_adversariales(X: np.ndarray,
                                    ruido_std: float = 0.05,
                                    seed: int = 42) -> np.ndarray:
    """
    Genera ejemplos adversariales añadiendo ruido gaussiano.
    En producción se usaría FGSM, PGD, etc.
    """
    rng   = np.random.default_rng(seed)
    ruido = rng.normal(0, ruido_std, X.shape)
    return np.clip(X + ruido, 0, 1)  # mantener rango normalizado


# 2. Ataque con diferentes niveles de ruido
niveles_ruido = [0.01, 0.05, 0.10, 0.20, 0.30]
resultados = []

for std in niveles_ruido:
    X_adv = generar_ejemplos_adversariales(X_test, ruido_std=std)
    acc   = accuracy_score(y_test, modelo.predict(X_adv))
    resultados.append({'ruido_std': std, 'precision': acc})
    print(f'  ruido_std={std:.2f}  →  Precisión: {acc:.4f}')

# Visualizar degradación
df_res = pd.DataFrame(resultados)
plt.figure(figsize=(8, 4))
plt.plot(df_res['ruido_std'], df_res['precision'], marker='o', color='red')
plt.axhline(acc_normal, linestyle='--', color='steelblue', label=f'Sin ataque ({acc_normal:.4f})')
plt.xlabel('Nivel de ruido (std)')
plt.ylabel('Precisión')
plt.title('Degradación del modelo bajo ataque adversarial')
plt.legend()
plt.tight_layout()
plt.savefig('adversarial_degradation.png', dpi=150)
plt.show()
print('Gráfico guardado: adversarial_degradation.png')

## 8.4 Defensa: entrenamiento adversarial

In [ ]:
# Usar ruido_std=0.1 como caso de ataque representativo
X_test_adv  = generar_ejemplos_adversariales(X_test, ruido_std=0.1)
acc_adversa = accuracy_score(y_test, modelo.predict(X_test_adv))
print(f'Precisión normal       : {acc_normal:.4f}')
print(f'Precisión bajo ataque  : {acc_adversa:.4f}')

# 3. Defensa: reentrenamiento con ejemplos adversariales
X_train_robusto = np.vstack([
    X_train,
    generar_ejemplos_adversariales(X_train, ruido_std=0.05)
])
y_train_robusto = np.concatenate([y_train, y_train])

modelo_robusto = RandomForestClassifier(n_estimators=200, random_state=42)
modelo_robusto.fit(X_train_robusto, y_train_robusto)

acc_robusto_normal = accuracy_score(y_test, modelo_robusto.predict(X_test))
acc_robusto_adv    = accuracy_score(y_test, modelo_robusto.predict(X_test_adv))

print(f'\n=== Comparativa ===')
print(f'Modelo original  — Normal: {acc_normal:.4f}  |  Bajo ataque: {acc_adversa:.4f}')
print(f'Modelo robusto   — Normal: {acc_robusto_normal:.4f}  |  Bajo ataque: {acc_robusto_adv:.4f}')

# Visualización comparativa
categorias = ['Normal', 'Bajo ataque']
original   = [acc_normal, acc_adversa]
robusto    = [acc_robusto_normal, acc_robusto_adv]

x = np.arange(len(categorias))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, original, width, label='Modelo original', color='steelblue')
bars2 = ax.bar(x + width/2, robusto,  width, label='Modelo robusto',  color='darkorange')

ax.set_ylabel('Precisión')
ax.set_title('Comparativa: modelo original vs. modelo robusto')
ax.set_xticks(x)
ax.set_xticklabels(categorias)
ax.set_ylim(0, 1.05)
ax.legend()

for bar in bars1 + bars2:
    ax.annotate(f'{bar.get_height():.3f}',
                xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points',
                ha='center', va='bottom')

plt.tight_layout()
plt.savefig('adversarial_defense.png', dpi=150)
plt.show()
print('Gráfico guardado: adversarial_defense.png')

## Resumen

- El modelo original pierde precisión significativa bajo ataque adversarial.
- El **entrenamiento adversarial** (incluir ejemplos perturbados en el entrenamiento) mejora la robustez.
- En producción se recomienda usar ataques más sofisticados como **FGSM** o **PGD** para el entrenamiento defensivo.
- También es importante monitorear la distribución de entradas en producción para detectar intentos de evasión.